# Week 1 — Understanding the 2023 data and developing research questions

This notebook supports the first week of the course. It does four things:

1. verifies the basic structure of the shared 2023 World Risk Poll file;
2. distinguishes real answers from special survey codes;
3. introduces the variable families relevant to the project; and
4. creates evidence for open-ended research-question brainstorming.

No model is fitted this week. The point is to understand what the data can and cannot answer.

## 1. Setup and reproducibility

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

ROOT = Path.cwd()
if not (ROOT / "23_wrp" / "WRP_2023.csv").exists() and (ROOT.parent / "23_wrp" / "WRP_2023.csv").exists():
    ROOT = ROOT.parent

RAW_DATA = ROOT / "23_wrp" / "WRP_2023.csv"
DICTIONARY = ROOT / "23_wrp" / "WRP_2023_data_dictionary.csv"
ARTIFACTS = ROOT / "course_artifacts"
ARTIFACTS.mkdir(exist_ok=True)

assert RAW_DATA.exists(), f"Cannot find {RAW_DATA}"
assert DICTIONARY.exists(), f"Cannot find {DICTIONARY}"

## 2. Load the raw data and verify its structure

In [ ]:
df = pd.read_csv(RAW_DATA, low_memory=False)

audit = pd.Series({
    "respondent rows": len(df),
    "raw columns": df.shape[1],
    "distinct non-missing ISO3 codes": df["COUNTRY_ISO3"].nunique(dropna=True),
    "duplicate respondent IDs": int(df["WPID_RANDOM"].duplicated().sum()),
    "survey years in file": ", ".join(map(str, sorted(df["Year"].dropna().unique()))),
}, name="value")
display(audit.to_frame())

assert len(df) == 146_910
assert df.shape[1] == 325
assert df["COUNTRY_ISO3"].nunique() == 142
assert df["WPID_RANDOM"].duplicated().sum() == 0

### Why these assertions matter

Assertions turn expectations into executable checks. If another file is loaded accidentally, the notebook stops instead of silently producing a different answer key.

## 3. Week 1 numerical quiz answers

In [ ]:
food_special_responses = int(df["WP20720"].isin([98, 99]).sum())

week01_quiz_answers = pd.DataFrame({
    "quiz item": [
        "W01-Q01 Respondent rows",
        "W01-Q02 Countries or territories",
        "W01-Q03 Raw variables",
        "W01-Q04 WP20720 responses coded 98 or 99",
    ],
    "answer": [
        len(df),
        df["COUNTRY_ISO3"].nunique(dropna=True),
        df.shape[1],
        food_special_responses,
    ],
})
display(week01_quiz_answers.style.hide(axis="index"))

assert food_special_responses == 844

**Answer to W01-Q05:** codes `98` and `99` mean “Don't know” and refusal. They must be recoded as missing before distributions or models are calculated. They are not high values on the worry scale.

## 4. The 13 features reserved for risk-profile discovery

In [ ]:
WORRY_FEATURES = {
    "WP20720": "Food worry",
    "WP20721": "Water worry",
    "WP20722": "Violent-crime worry",
    "WP20723": "Severe-weather worry",
    "WP22213": "Traffic worry",
    "WP20726": "Mental-health worry",
}
EXPERIENCE_FEATURES = {
    "WP22442": "Food harm",
    "WP22443": "Water harm",
    "WP22444": "Violent-crime harm",
    "WP22445": "Severe-weather harm",
    "WP22446": "Traffic harm",
    "WP22447": "Mental-health harm",
    "WP22448": "Work harm",
}
PROFILE_FEATURES = list(WORRY_FEATURES) + list(EXPERIENCE_FEATURES)

feature_map = pd.DataFrame({
    "variable": PROFILE_FEATURES,
    "meaning": list(WORRY_FEATURES.values()) + list(EXPERIENCE_FEATURES.values()),
    "role": ["ordered worry"] * len(WORRY_FEATURES) + ["experience of harm"] * len(EXPERIENCE_FEATURES),
})
display(feature_map)
print(f"Profile-discovery features: {len(PROFILE_FEATURES)}")
assert len(PROFILE_FEATURES) == 13

Climate threat (`WP20719`) and conditional workplace worry (`WP22214`) are not included in this clustering matrix. Climate threat uses a different concept, while workplace worry is unavailable by design for respondents outside the eligible employment group. Both remain available for later, question-specific analyses.

## 5. Special codes and usable responses

In [ ]:
special_code_table = pd.DataFrame(index=PROFILE_FEATURES)
special_code_table["code_98_n"] = [int((df[c] == 98).sum()) for c in PROFILE_FEATURES]
special_code_table["code_99_n"] = [int((df[c] == 99).sum()) for c in PROFILE_FEATURES]
special_code_table["ordinary_missing_n"] = [int(df[c].isna().sum()) for c in PROFILE_FEATURES]
special_code_table["usable_n"] = [int((~df[c].isin([98, 99]) & df[c].notna()).sum()) for c in PROFILE_FEATURES]
special_code_table["meaning"] = [
    *(WORRY_FEATURES[c] for c in WORRY_FEATURES),
    *(EXPERIENCE_FEATURES[c] for c in EXPERIENCE_FEATURES),
]
display(special_code_table[["meaning", "code_98_n", "code_99_n", "ordinary_missing_n", "usable_n"]])

## 6. First descriptive view

In [ ]:
worry = df[list(WORRY_FEATURES)].apply(pd.to_numeric, errors="coerce").mask(lambda x: x.isin([98, 99]))
worry_pct = pd.concat(
    {WORRY_FEATURES[c]: worry[c].value_counts(normalize=True).sort_index().mul(100) for c in WORRY_FEATURES},
    axis=1,
).T.rename(columns={1: "very worried", 2: "somewhat worried", 3: "not worried"})
display(worry_pct.round(1))

ax = worry_pct[["very worried", "somewhat worried", "not worried"]].plot(
    kind="barh", stacked=True, figsize=(10, 5), color=["#c44e52", "#ddcc77", "#55a868"]
)
ax.set(xlabel="Percent among substantive responses", ylabel="", xlim=(0, 100), title="Worry differs across risk domains")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
plt.tight_layout()
plt.show()

## 7. Open-ended research-question brainstorming

The numerical audit has fixed answers. Research questions do not.

For the two free-text quiz items, propose at least three questions and then select one provisional favorite. Useful directions include discovering patterns among respondents, predicting an outcome, comparing contexts, or testing whether a finding generalizes. Do not begin with an algorithm name. Begin with what you want to discover.

For every proposal, state:

- what we would learn;
- why the answer matters;
- which variable families could provide evidence; and
- one limitation or alternative explanation.

## Week 1 conclusion

The current harmonised file contains 146,910 respondents, 142 countries or territories, and 325 raw columns. It is large and globally diverse, but its survey codes and conditional items require semantic cleaning. The shared profile-discovery block contains 13 features, while demographics and country context are reserved for later objectives.